In [109]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import colour
import json
import base64
import os
from ipywidgets import Button, HBox, VBox, HTML, Layout, FileUpload, Output, Text, FloatSlider
from IPython.display import display, Javascript

WAVE_MIN, WAVE_MAX = 380, 780
WAVE_SAMPLES = np.arange(WAVE_MIN, WAVE_MAX + 1, 1)

class RobustSpectralTool:
    def __init__(self):
        self.points = {380: 50.0, 780: 50.0}
        self.dragging_key = None
        self.pick_radius = 8
        self.output_log = Output()
        
        # --- UI Elements ---
        self.color_tile = HTML(value="")
        
        # Path Textbox initialization
        default_path = os.path.join(os.getcwd(), "spectrum_points.json")
        self.path_box = Text(
            value=default_path,
            placeholder='Enter save path...',
            description='Save Path:',
            layout=Layout(width='90%'),
            style={'description_width': 'initial'}
        )
        
        self.export_btn = Button(description="Export JSON", button_style='success', icon='download')
        self.export_btn.on_click(self.handle_export)
        
        self.import_uploader = FileUpload(accept='.json', multiple=False, description="Import JSON")
        self.import_uploader.observe(self.import_from_browser, names='value')
        
        self.clear_btn = Button(description="Reset", button_style='danger', icon='trash')
        self.clear_btn.on_click(lambda b: self.reset())

        # --- Inside RobustSpectralTool.__init__ ---
        self.background_uploader = FileUpload(accept='image/*', description="BG Image", button_style='info')
        self.background_uploader.observe(self.handle_image_upload, names='value')
        
        self.clear_bg_btn = Button(description="Clear BG", button_style='warning', icon='eraser')
        self.clear_bg_btn.on_click(self.handle_clear_bg)
        
        # Image geometry sliders
        slider_layout = Layout(width='300px')
        self.img_x = FloatSlider(value=WAVE_MIN, min=WAVE_MIN-200, max=WAVE_MAX, description='X Pos:', layout=slider_layout)
        self.img_y = FloatSlider(value=0, min=-50, max=100, description='Y Pos:', layout=slider_layout)
        self.img_w = FloatSlider(value=WAVE_MAX-WAVE_MIN, min=10, max=1000, description='Width:', layout=slider_layout)
        self.img_h = FloatSlider(value=100, min=10, max=300, description='Height:', layout=slider_layout)
        
        for s in [self.img_x, self.img_y, self.img_w, self.img_h]:
            s.observe(self.update_image_geometry, names='value')
        
        self.bg_image_obj = None

        
        # Pre-cache CMFs
        self.cmfs = colour.MSDS_CMFS['CIE 1931 2 Degree Standard Observer'].copy().align(
            colour.SpectralShape(WAVE_MIN, WAVE_MAX, 1))
        self.illuminant = colour.SDS_ILLUMINANTS['D65'].copy().align(self.cmfs.shape)
        
        # Setup Figure
        self.fig, self.ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
        self.fig.canvas.header_visible = False
        self.line, = self.ax.plot([], [], color='#444', lw=1.5, animated=True)
        self.dots = self.ax.scatter([], [], color='black', s=35, zorder=4, animated=True, edgecolors='white', linewidths=0.5)
        
        gradient = np.linspace(WAVE_MIN, WAVE_MAX, 256)
        self.ax.imshow([gradient], extent=[WAVE_MIN, WAVE_MAX, -5, 0], aspect='auto', cmap='turbo', zorder=0)
        self.ax.set_ylim(-5, 105)
        self.ax.set_xlim(WAVE_MIN - 20, WAVE_MAX + 20)
        self.ax.grid(True, linestyle=':', alpha=0.3)
        
        self.fig.canvas.mpl_connect('button_press_event', self.on_click)
        self.fig.canvas.mpl_connect('button_release_event', self.on_release)
        self.fig.canvas.mpl_connect('motion_notify_event', self.on_move)
        self.fig.canvas.mpl_connect('draw_event', self.on_draw)

        self.bg = None
        self.update_calculations(full_metrics=True)

    def update_calculations(self, full_metrics=False):
        sorted_keys = sorted(self.points.keys())
        sorted_vals = [self.points[k] for k in sorted_keys]
        l_bound, r_bound = float(sorted_vals[0]), float(sorted_vals[-1])
        full_y = np.interp(WAVE_SAMPLES, sorted_keys, sorted_vals, left=l_bound, right=r_bound)
        
        self.line.set_data(WAVE_SAMPLES, full_y)
        self.dots.set_offsets(np.c_[list(self.points.keys()), list(self.points.values())])
        self.dots.set_color(['red' if k == self.dragging_key else 'black' for k in self.points.keys()])
        
        sd = colour.SpectralDistribution(dict(zip(WAVE_SAMPLES, full_y / 100.0)))
        XYZ = colour.sd_to_XYZ(sd, self.cmfs, self.illuminant) / 100.0
        Lab = colour.XYZ_to_Lab(XYZ)
        rgb = np.clip(colour.XYZ_to_sRGB(XYZ), 0, 1)
        hex_c = '#%02x%02x%02x' % tuple((rgb * 255).astype(int))
        
        m_str = ""
        if full_metrics:
            try: m_str = f" | Munsell: {colour.xyY_to_munsell_colour(colour.XYZ_to_xyY(XYZ))}"
            except: m_str = " | Munsell: N/A"
        
        text_color = "white" if Lab[0] < 50 else "black"
        self.color_tile.value = f'<div style="background:{hex_c}; padding:10px; border-radius:5px; text-align:center; color:{text_color}; font-family:sans-serif; border:1px solid #999;"><b>{hex_c.upper()}</b> — L*{Lab[0]:.1f} a*{Lab[1]:.1f} b*{Lab[2]:.1f}{m_str}</div>'

    def handle_export(self, b):
        self.output_log.clear_output()
        target_path = self.path_box.value
        
        sorted_keys = sorted(self.points.keys())
        data = {"wavelengths": [round(float(k), 2) for k in sorted_keys], 
                "reflectance": [round(float(self.points[k]), 2) for k in sorted_keys]}
        
        try:
            # Local Save to the specified path
            with open(target_path, "w") as f:
                json.dump(data, f)
            with self.output_log:
                print(f"Successfully saved to: {os.path.abspath(target_path)}")
        except Exception as e:
            with self.output_log:
                print(f"Export Error (Local Save): {e}")

        # Fallback Browser Download
        b64 = base64.b64encode(json.dumps(data).encode()).decode()
        js = f"const a=document.createElement('a');a.href='data:application/json;base64,{b64}';a.download='{os.path.basename(target_path)}';a.click();"
        display(Javascript(js))

    def import_from_browser(self, change):
        if not change['new']: return
        try:
            # ipywidgets 8.x: value is a list of dicts
            uploaded_file = change['new'][0]
            raw = uploaded_file['content']
            if hasattr(raw, 'tobytes'): raw = raw.tobytes()
            content = json.loads(raw.decode('utf-8'))
            
            self.points = dict(zip(content['wavelengths'], content['reflectance']))
            self.update_calculations(True)
            self.fig.canvas.draw_idle()
        except Exception as e:
            with self.output_log: print(f"Import Error: {e}")
        self.import_uploader.value = []

    def handle_image_upload(self, change):
        if not change['new']: return
        import io
        from PIL import Image
        try:
            # Support for ipywidgets 8.x
            uploaded_file = change['new'][0]
            img = Image.open(io.BytesIO(uploaded_file['content']))
            
            if self.bg_image_obj:
                self.bg_image_obj.remove()
                
            # Initial draw with current slider values
            self.bg_image_obj = self.ax.imshow(
                img, aspect='auto', origin='upper', alpha=0.5, zorder=0.1
            )
            self.update_image_geometry(None)
        except Exception as e:
            with self.output_log: print(f"Image Error: {e}")
    
    def update_image_geometry(self, change):
        """Calculates extent: [left, right, bottom, top] based on sliders."""
        if self.bg_image_obj:
            left = self.img_x.value
            right = left + self.img_w.value
            bottom = -self.img_y.value
            top = bottom + self.img_h.value
            
            self.bg_image_obj.set_extent([left, right, bottom, top])
            self.bg = None # Force background cache reset
            self.fig.canvas.draw_idle()

    
    def handle_clear_bg(self, b):
        """Removes the image and resets the uploader widget."""
        if self.bg_image_obj:
            self.bg_image_obj.remove()
            self.bg_image_obj = None
        self.background_uploader.value = []  # Clear uploader
        self.fig.canvas.draw_idle()



    # --- Interaction Boilerplate ---
    def on_draw(self, event): 
        self.bg = self.fig.canvas.copy_from_bbox(self.ax.bbox)
        self.draw_artists()

    def draw_artists(self):
        if self.bg: 
            self.fig.canvas.restore_region(self.bg)
            # If image exists, it's already in the background 'bg' if 
            # captured after upload. For safety, let's just use draw_idle 
            # for image updates and blit for point dragging.
            self.ax.draw_artist(self.line)
            self.ax.draw_artist(self.dots)
            self.fig.canvas.blit(self.ax.bbox)

    def get_closest(self, event):
        if not self.points: return None, 999
        display_pts = self.ax.transData.transform(np.c_[list(self.points.keys()), list(self.points.values())])
        dists = np.linalg.norm(display_pts - np.array([event.x, event.y]), axis=1)
        idx = np.argmin(dists)
        return list(self.points.keys())[idx], dists[idx]

    def on_click(self, event):
        # --- ADD THIS CHECK ---
        # If zoom or pan is active, do nothing
        if self.fig.canvas.toolbar.mode != "":
            return
        if event.inaxes != self.ax: return
        key, dist = self.get_closest(event)
        if event.button == 3 and dist < self.pick_radius: del self.points[key]
        elif event.button == 1:
            if dist < self.pick_radius: self.dragging_key = key
            else: self.points[event.xdata] = np.clip(event.ydata, 0, 100); self.dragging_key = event.xdata
        self.update_calculations(); self.draw_artists()

    def on_move(self, event):
        # --- ADD THIS CHECK ---
        # If zoom or pan is active, do nothing
        if self.fig.canvas.toolbar.mode != "":
            return
        if self.dragging_key is None or event.inaxes != self.ax: return
        new_x, new_y = np.clip(event.xdata, WAVE_MIN, WAVE_MAX), np.clip(event.ydata, 0, 100)
        self.points.pop(self.dragging_key); self.points[new_x] = new_y; self.dragging_key = new_x
        self.update_calculations(False); self.draw_artists()

    def on_release(self, event): 
        self.dragging_key = None; self.update_calculations(True); self.draw_artists(); self.fig.canvas.draw_idle()

    def reset(self): 
        self.points = {380: 50.0, 780: 50.0}; self.update_calculations(True); self.fig.canvas.draw_idle()

plt.ioff()
tool = RobustSpectralTool()

# Group sliders for better layout
slider_box = VBox([
    HBox([tool.img_x, tool.img_w]),
    HBox([tool.img_y, tool.img_h])
], layout=Layout(margin='10px 0'))

display(VBox([
    tool.fig.canvas, 
    tool.color_tile, 
    HBox([tool.export_btn, tool.import_uploader, tool.clear_btn], layout=Layout(justify_content='center', grid_gap='10px')),
    HBox([tool.background_uploader, tool.clear_bg_btn], layout=Layout(justify_content='center', grid_gap='10px')),
    slider_box,
    tool.path_box,
    tool.output_log
], layout=Layout(align_items='center')))
